# Chapter 2: What Is an LLM?

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch02-what-is-an-llm/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch02-what-is-an-llm.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 2: What Is an LLM?

![The map highlights the whole chain as an overview](../assets/diagrams/ch02-where-we-are.png){ width="756" }
*Figure 2.1: Before we zoom in, let's look at what the whole system actually does.*

With our environment set up in the previous chapter, we are ready to understand the model itself (Figure 2.1). A Language Model (LM) seems like a magic box that understands what you type. But under the hood, it performs a very specific, simple task.

In this chapter you will:

- Learn the single rule that drives all language models.
- See how models generate long answers one step at a time.
- Understand how reading data replaces human teachers.

**Words to Know**
    - **Token**: one small piece of text, here a single character.
    - **Language Model**: a system that predicts the next token in a sequence.
    - **Autoregressive**: using your past outputs as inputs for your next step.
    - **Parameters**: the numbers inside the model that adjust during training.

## Theory: The Next-Token Engine

Imagine you are texting a friend and you type: "I am so hungry, I could eat a". Before you finish, your phone suggests: **horse**. Or **pizza**.

![A text box with incomplete text points to a prediction box showing horse](../assets/diagrams/ch02-next-word.png){ width="318" }
*Figure 2.2: The core task of a language model is guessing what comes next.*

As Figure 2.2 shows, your phone is doing **next-word prediction**. It guesses what word is most likely to come next based on everything you typed so far. A Large Language Model (LLM) does exactly the same thing. Given a sequence of tokens, it predicts what comes next. That is the whole idea. Everything else in this book is just building a machine that can do this really well.

### How It Talks Back

If the model only predicts the next token, how does it write an essay? It uses **autoregressive generation**.

1. You type: `"What is the capital of France?"`
2. The model predicts the next token: `"Paris"`.
3. That output is added to the input: `"What is the capital of France? Paris"`.
4. It predicts the next token: `" is"`.
5. And repeats until it decides to stop.

![A diagram showing the output text being fed back into the model as input](../assets/diagrams/ch02-autoregressive.png){ width="458" }
*Figure 2.3: In autoregressive generation, each predicted output loops back to become the next input.*

As illustrated in Figure 2.3, the model never thinks about the whole answer at once. It just keeps predicting the next piece, over and over, building the sentence step by step.

### What is a Token?

A token is the smallest unit the model works with. It could be a whole word, a piece of a word, or just a single character.

In this book, we use **character-level tokens**. Every single letter, space, and punctuation mark is one token. We use characters because the vocabulary is tiny (only 65 characters in our Shakespeare data) and you can understand it instantly. Real models use more complex chunks (subwords), but the math is exactly the same.

### The Power of Self-Supervised Learning

To make a model "Large", we give it millions of parameters, which are the numbers inside the model that act like dials. Training turns these dials until the model produces good output.

But how does it learn without a teacher grading its work? Through **self-supervised learning**.

With language, the text itself is the answer key. If your training text is "Hello world", the model gets these practice questions automatically:

- See `H`, predict `e`.
- See `He`, predict `l`.
- See `Hel`, predict `l`.

![A diagram showing text subsets predicting the next character](../assets/diagrams/ch02-self-supervised.png){ width="318" }
*Figure 2.4: The text itself provides thousands of built-in practice questions.*

As Figure 2.4 demonstrates, because no human labels are needed, you can train a model on millions of pages of raw text. The model just reads the data and learns the patterns.

## Try It: The Numbers Game

Is it really just predicting characters? Let's prove it by looking at real data. If we see the letters `"the "`, what is the most likely next character in Shakespeare?

We wrote a tiny script to scan our training data and count every character that follows `"the "`.

```python
$ python src/examples/ch02_next_char.py
What follows 'the ' in Shakespeare?
's': 573 times
'w': 459 times
'c': 448 times
'p': 426 times
'm': 376 times
```

This is exactly what the model learns to do, but instead of counting by hand, it uses math to predict the probabilities.

**In Business**
    If a company builds an AI assistant to draft emails in its own house style, the concept is the same. The model trains on the company's archive of past writing. By learning what character or word typically follows another in that specific archive, the assistant learns the company's unique voice and terminology automatically.

**Watch Out**
    It is easy to think the model "understands" the text. It does not. It is simply a math engine calculating the most probable next token based on patterns in its training data.

## Key Takeaways

- A language model is a system that predicts the next token.
- Autoregressive generation means the model uses its own outputs as inputs for the next step.
- Tokens are the basic pieces of text, like characters or words.
- Self-supervised learning uses the raw text itself as the answer key.
- Now that we know the whole system revolves around next-token prediction, let's zoom in on the first step in our map: the math and tools we use to build it.

## Check Your Understanding
1. What does a language model actually predict?
2. Why is it called "autoregressive"?
3. How does self-supervised learning differ from having a teacher grade the work?


## Further Reading

**One model, many jobs, no retraining.** The question was whether a model trained only to predict the next word would pick up skills nobody trained it for. Trained on a large, varied sweep of web pages, it began answering questions, summarizing and translating with no task-specific training at all, simply because the prompt made the task clear. That settled the architecture question for text generation: a decoder that predicts the next token, which is the model in Chapter 10.

**Memory, before attention.** Models that read a sentence one word at a time kept forgetting the beginning by the time they reached the end, because the learning signal faded as it travelled back through the steps. The fix was a cell with gates that decide what to keep, what to drop, and what to pass on. This ran almost every serious language system for twenty years. The question it answers, what should I still remember from earlier in the text, is the same question attention answers, by a completely different route.

<div class="refs" markdown>

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735

Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language models are unsupervised multitask learners* [Technical report]. OpenAI. https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

</div>

---

### `src/examples/ch02_next_char.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch02_next_char.py"   # a cell has none, and the file uses it to find the text

"""Shows what character follows the string 'the ' in Shakespeare."""
import os
import sys
from collections import Counter

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

# Read the dataset
data_path = os.path.join(os.path.dirname(__file__), "..", "data", "shakespeare.txt")
with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

# Find what follows "the "
prefix = "the "
next_chars = []
for i in range(len(text) - len(prefix)):
    if text[i:i+len(prefix)].lower() == prefix:
        next_chars.append(text[i+len(prefix)])

# Count and print the top 5
counts = Counter(next_chars)
print(f"What follows '{prefix}' in Shakespeare?")
for char, count in counts.most_common(5):
    display_char = repr(char) if char == ' ' or char == '\n' else char
    print(f"'{display_char}': {count} times")